# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cpu


In [4]:
from mllm_shap.connectors import TransformersCausalText, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import PowerShiftNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define a `TransformersCausalText` model (this call loads it into memory!).

Create a compact explainer using `McShapExplainer` with `num_samples=-1` (enumerate all possible masks) and contextual embeddings. `PowerShiftNormalizer(power=2.0)` shifts values so the minimum is 0, raises them to the power of 2, then normalizes so they sum to 1.

In [5]:
model = TransformersCausalText(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = McShapExplainer(
    num_samples=-1,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=PowerShiftNormalizer(
        power=2.0
    ),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

Create a new chat with `SystemRolesSetup.SYSTEM_ASSISTANT` — assistant turns are treated as fixed system context and are not masked during SHAP evaluation. This significantly reduces the number of evaluations needed for multi-turn chats. `KeepAllTokens` keeps every token (including punctuation) in scope for SHAP value calculation.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,
    token_filter=KeepAllTokens(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.SYSTEM)
chat.add_text("You are a helpful assistant that answers questions briefly.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

Let's have a look at chat representation:

In [7]:
chat.get_conversation()

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=None)],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 64 tokens and change text_temperature from default 0.0 to 0.2. 

In [8]:
generation_kwargs = {
    "max_new_tokens": 64,
    "model_config": ModelConfig(text_temperature=0.2),
}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2026-05-05 13:34:54,339 - mllm_shap.shap.compact - INFO - Generating full response from the model...
/Users/pawel.pozorski/Desktop/MLLM-Shap/mllm_shap/src/mllm_shap/shap/compact.py:60: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2026-05-05 13:34:57,436 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 4 (up to 15 additional calls)


Monte Carlo SHAP:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/pawel.pozorski/Desktop/MLLM-Shap/mllm_shap/src/mllm_shap/shap/base/_generate_responses.py:237: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  model_response = model.generate(
2026-05-05 13:35:02,474 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=5 cache_hits=0 cache_misses=5 skipped_filtered=0 model_elapsed_ms=5021.34
2026-05-05 13:35:02,475 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=8 yielded=5 skipped(full_or_empty)=2 skipped(invalid)=0 skipped(duplicates)=1 elapsed_ms=5025.34


Result has now following fields available:

- full_chat - chat with base response (generated based on whole entry) with set cache and calculated shapley values
- source_chat - original chat feed to the explainer
- history - history of all chats used

Cache object stores actual shapley values as well as calculated embeddings and masks. They will be reused in next call regardless to the method, so for monte-carlo it is just larger sample, for precise it means some results might get excluded.

Let's first analyze history - it is a list of size equivalent to the number of non-trivial mask evaluations. With `KeepAllTokens` all user tokens are explained, so the count equals 2ⁿ − 2 (excluding the all-True full prompt and the all-False empty prompt). Each entry is a tuple of the following values:

- mask for that entry
- mash hash
- source chat with masked entry or None if corresponding mask was available in cache
- model response object

or None - when either corresponding mask was extracted from cache or it has risen an AllTextTokensFilteredOutError error.

Let's see all chats that were taken into account:

In [9]:
[c[2].decode_text() if c is not None else None for c in result.history]

['You are a helpful assistant that answers questions briefly. are you?',
 'You are a helpful assistant that answers questions briefly.Who you?',
 'You are a helpful assistant that answers questions briefly.Who are?',
 'You are a helpful assistant that answers questions briefly.Who are you',
 'You are a helpful assistant that answers questions briefly. are?']

All tokens in the user turn (including `?`) are explainable because `KeepAllTokens` is used — any token can be masked. System and assistant turns are always present across all masked queries because `SystemRolesSetup.SYSTEM_ASSISTANT` treats them as fixed context.

Let's now analyze calculated shapley values.

In [10]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=[0.02980850450694561, 0.0, 0.9074166417121887, 0.06277485191822052])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Answer, :,  I,  am,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ., \n, <|en...', shap_values=[nan, nan, ..., nan, nan])]]


The model tracks only text tokens. `shap_values` is now populated for all user tokens. Tokens from system/assistant turns have `NaN` values as they are outside the explanation scope. With `KeepAllTokens` all user tokens are explained (no `NaN` among user tokens). Let's display them.

In [11]:
user_entry = explained_chat_conversation[1][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,Who,0.029809,0
1,are,0.000000,0
2,you,0.907417,0
3,?,0.062775,0


Let's create another turn to see how input significance will change:

In [12]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

And again, let's explain it:

In [13]:
result = explainer(
    chat=explained_chat, verbose=True, generation_kwargs=generation_kwargs
)

2026-05-05 13:35:02,600 - mllm_shap.shap.compact - INFO - Generating full response from the model...
/Users/pawel.pozorski/Desktop/MLLM-Shap/mllm_shap/src/mllm_shap/shap/compact.py:60: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  response = self.model.generate(
2026-05-05 13:35:04,182 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 8 (up to 255 additional calls)


Monte Carlo SHAP:   0%|          | 0/9 [00:00<?, ?it/s]

/Users/pawel.pozorski/Desktop/MLLM-Shap/mllm_shap/src/mllm_shap/shap/base/_generate_responses.py:237: UserWarning: Audio generation parameters were provided but this connector is text-only;                     audio settings are ignored.
  model_response = model.generate(
2026-05-05 13:35:20,603 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=9 cache_hits=0 cache_misses=9 skipped_filtered=0 model_elapsed_ms=16413.75
2026-05-05 13:35:20,604 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=10 yielded=9 skipped(full_or_empty)=1 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=16419.19


In [14]:
[c[2].decode_text() if c is not None else None for c in result.history]

['You are a helpful assistant that answers questions briefly. are you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are you\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>Can you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|> you repeat?',
 'You are a helpful assistant that answers questions briefly.Who are you?\nAnswer: I am a helpful assistant that answers questions briefly.\n<|endoftext|>C

In [15]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ....', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Who,  are,  you, ?', shap_values=[0.13872495293617249, 0.10585115104913712, 0.13769467175006866, 0.08477983623743057])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, Answer, :,  I,  am,  a,  helpful,  assistant,  that,  answers,  questions,  briefly, ., \n, <|en...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Can,  you,  repeat, ?', shap_values=[0.17841790616512299, 0.0, 0.24367520213127136, 0.11085628718137741])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content=' I,  said,  to,  turn,  off,  the,  lights,  before,  leaving, .,  , \n, A, :,  I,  said,  t

In [16]:
dt = []
for i in (1, 3):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,Who,0.138725,0,1
1,are,0.105851,0,1
2,you,0.137695,0,1
3,?,0.084780,0,1
4,Can,0.178418,0,3
5,you,0.000000,0,3
6,repeat,0.243675,0,3
7,?,0.110856,0,3
